<a href="https://colab.research.google.com/github/goelnikhils-lgtm/languagemodels/blob/main/Production_Grade_Privacy_Preserving_Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Production-Grade Privacy-Preserving Encoder
============================================
Base model: sentence-transformers/all-MiniLM-L12-v2
            (12-layer, 384-dim, 33M params, contrastively pretrained on 1B+ pairs)

Upgrades over earlier versions
------------------------------
1. Stronger backbone (MiniLM-L12-v2) with proper sentence-transformers
   mean-pooling AND L2-normalization.
2. LoRA-friendly attention adaptation with sane defaults.
3. Tiered multi-label adversary heads (scales to 100+ attributes).
4. MINE-style mutual-information upper bound on I(z; sensitive_vector).
5. Variational information bottleneck (VIB) with KL annealing.
6. Two-loop adversarial training (encoder vs MINE) with separate optimizers.
7. Release-time Gaussian DP mechanism with calibrated sensitivity bound
   via L2-normalization of z.
8. Per-tier audit reports with severity tiers AND statistical CIs.
9. EMA of encoder weights for stable deployment checkpoints.
10. Clean public API: encode_for_release(texts, epsilon, delta) -> np.ndarray

Requirements
------------
  pip install torch transformers numpy scikit-learn
  # Optional, recommended:
  pip install peft        # LoRA
  pip install opacus      # DP-SGD (training-time DP)
"""

from __future__ import annotations
import math
import copy
from dataclasses import dataclass, field
from typing import Literal, Optional, Iterable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import DataLoader, Dataset

from transformers import AutoTokenizer, AutoModel


# ═════════════════════════════════════════════════════════════════
#  CONFIGURATION DATA CLASSES
# ═════════════════════════════════════════════════════════════════

@dataclass
class Attribute:
    """A single sensitive attribute to scrub."""
    name: str
    n_classes: int = 2
    kind: Literal["binary", "categorical", "continuous"] = "binary"
    pos_weight: float = 1.0     # for class-imbalanced binary attrs


@dataclass
class SensitivityTier:
    """Group of attributes scrubbed with a shared adversarial weight."""
    name: str
    weight: float
    attributes: list[Attribute] = field(default_factory=list)

    @property
    def n_binary(self) -> int:
        return sum(1 for a in self.attributes if a.kind == "binary")

    @property
    def categorical_attrs(self) -> list[Attribute]:
        return [a for a in self.attributes if a.kind == "categorical"]

    @property
    def continuous_attrs(self) -> list[Attribute]:
        return [a for a in self.attributes if a.kind == "continuous"]


@dataclass
class PrivacyConfig:
    """Hyperparameters for privacy training."""
    z_dim: int = 96
    proj_hidden: int = 384
    adv_hidden: int = 192
    mine_hidden: int = 384
    mode: Literal["head", "lora", "full"] = "lora"
    use_mine: bool = True
    use_ib: bool = True
    lambda_adv_max: float = 1.0
    lambda_kl_max: float = 1e-2
    lambda_mine: float = 0.05
    kl_anneal_epochs: int = 5
    mine_steps_per_batch: int = 2
    grad_clip: float = 1.0
    weight_decay: float = 1e-4
    lr_main: float = 3e-4
    lr_mine: float = 5e-4
    ema_decay: float = 0.999


# ═════════════════════════════════════════════════════════════════
#  GRADIENT REVERSAL
# ═════════════════════════════════════════════════════════════════

class _GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad):
        return grad.neg() * ctx.lambda_, None


def grad_reverse(x: torch.Tensor, lambda_: float = 1.0) -> torch.Tensor:
    return _GradReverse.apply(x, lambda_)


# ═════════════════════════════════════════════════════════════════
#  BACKBONE: all-MiniLM-L12-v2
# ═════════════════════════════════════════════════════════════════

DEFAULT_MODEL = "sentence-transformers/all-MiniLM-L12-v2"


class MiniLMBackbone(nn.Module):
    """
    Wraps sentence-transformers/all-MiniLM-L12-v2.
    - Uses attention-aware mean pooling (the official ST recipe).
    - Optionally L2-normalizes (also the official ST recipe; needed for the
      DP sensitivity bound to be exactly 2.0 between any two embeddings).
    - Three training modes: 'head' (frozen), 'lora', 'full'.
    """

    def __init__(self,
                 model_name: str = DEFAULT_MODEL,
                 mode: Literal["head", "lora", "full"] = "lora",
                 normalize: bool = True,
                 max_length: int = 256):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size      # 384 for L12-v2
        self.normalize = normalize
        self.max_length = max_length
        self.mode = mode

        if mode == "head":
            for p in self.encoder.parameters():
                p.requires_grad = False
        elif mode == "lora":
            try:
                from peft import LoraConfig, get_peft_model
            except ImportError as e:
                raise ImportError(
                    "Install peft for LoRA mode: pip install peft"
                ) from e
            for p in self.encoder.parameters():
                p.requires_grad = False
            lora_cfg = LoraConfig(
                r=8, lora_alpha=16, lora_dropout=0.05,
                target_modules=["query", "key", "value"],
                bias="none",
            )
            self.encoder = get_peft_model(self.encoder, lora_cfg)
        # 'full' leaves everything trainable

    @staticmethod
    def _mean_pool(token_embeddings: torch.Tensor,
                   attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(token_embeddings)
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, texts: list[str], device: str = "cpu") -> torch.Tensor:
        enc = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=self.max_length, return_tensors="pt",
        ).to(device)
        out = self.encoder(**enc)
        pooled = self._mean_pool(out.last_hidden_state, enc["attention_mask"])
        if self.normalize:
            pooled = F.normalize(pooled, p=2, dim=1)
        return pooled


# ═════════════════════════════════════════════════════════════════
#  VARIATIONAL PROJECTION HEAD (information bottleneck)
# ═════════════════════════════════════════════════════════════════

class VariationalProjection(nn.Module):
    def __init__(self, in_dim: int, hidden: int, z_dim: int):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.GELU(), nn.LayerNorm(hidden),
        )
        self.mu_head = nn.Linear(hidden, z_dim)
        self.logvar_head = nn.Linear(hidden, z_dim)

    def forward(self, e: torch.Tensor):
        h = self.trunk(e)
        mu = self.mu_head(h)
        logvar = self.logvar_head(h).clamp(-8, 8)
        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        std = (0.5 * logvar).exp()
        return mu + std * torch.randn_like(std)


# ═════════════════════════════════════════════════════════════════
#  TIER ADVERSARY HEADS
# ═════════════════════════════════════════════════════════════════

class TierAdversaryHead(nn.Module):
    """
    One adversary per sensitivity tier:
      - 2-layer shared trunk
      - SINGLE multi-label binary head for all binary attrs in the tier
      - One linear head per categorical / continuous attribute
    Replaces dozens of independent MLPs with one fused module per tier.
    """

    def __init__(self, z_dim: int, tier: SensitivityTier, hidden: int = 192):
        super().__init__()
        self.tier = tier
        self.trunk = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.GELU(), nn.LayerNorm(hidden),
        )
        self.binary_head = (nn.Linear(hidden, tier.n_binary)
                            if tier.n_binary > 0 else None)
        self.cat_heads = nn.ModuleDict({
            a.name: nn.Linear(hidden, a.n_classes)
            for a in tier.categorical_attrs
        })
        self.cont_heads = nn.ModuleDict({
            a.name: nn.Linear(hidden, 1)
            for a in tier.continuous_attrs
        })

    def forward(self, z: torch.Tensor) -> dict:
        h = self.trunk(z)
        out = {}
        if self.binary_head is not None:
            out["binary"] = self.binary_head(h)
        out["categorical"] = {n: head(h) for n, head in self.cat_heads.items()}
        out["continuous"] = {n: head(h).squeeze(-1)
                             for n, head in self.cont_heads.items()}
        return out


# ═════════════════════════════════════════════════════════════════
#  MINE: mutual-information bound
# ═════════════════════════════════════════════════════════════════

class MINE(nn.Module):
    """
    Mutual Information Neural Estimation (Belghazi et al. 2018).
    Lower bound on I(Z; A) via the Donsker–Varadhan representation:
        I_DV(Z; A) ≥ E_p[T] − log E_q[exp(T)]
    Maximize wrt MINE params; encoder minimizes the same value to reduce I(Z; A).
    """

    def __init__(self, z_dim: int, a_dim: int, hidden: int = 384):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + a_dim, hidden), nn.ELU(),
            nn.Linear(hidden, hidden), nn.ELU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, z: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        t_joint = self.net(torch.cat([z, a], dim=-1))
        a_shuf = a[torch.randperm(a.size(0), device=a.device)]
        t_marg = self.net(torch.cat([z, a_shuf], dim=-1))
        # Numerically stable DV bound
        log_mean_exp = torch.logsumexp(t_marg, dim=0) - math.log(t_marg.size(0))
        return t_joint.mean() - log_mean_exp


# ═════════════════════════════════════════════════════════════════
#  EXPONENTIAL MOVING AVERAGE (for stable inference)
# ═════════════════════════════════════════════════════════════════

class EMA:
    """Tracks an EMA of model parameters; used for the deployed encoder."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters()
                       if p.requires_grad}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.detach(),
                                                     alpha=1 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        for n, p in model.named_parameters():
            if n in self.shadow:
                p.copy_(self.shadow[n])


# ═════════════════════════════════════════════════════════════════
#  FULL MODEL
# ═════════════════════════════════════════════════════════════════

class PrivacyEncoder(nn.Module):
    """
    Production privacy-preserving encoder over MiniLM-L12-v2.

    Public API:
        encode_for_release(texts, epsilon=2.0, delta=1e-5) -> np.ndarray
        encode(texts) -> Tensor                 (raw, training-only)
        forward(texts, lambda_adv) -> dict      (used internally by training)
    """

    def __init__(self,
                 tiers: list[SensitivityTier],
                 n_task_classes: int = 2,
                 model_name: str = DEFAULT_MODEL,
                 cfg: Optional[PrivacyConfig] = None):
        super().__init__()
        self.cfg = cfg or PrivacyConfig()
        self.tiers = tiers
        self.backbone = MiniLMBackbone(model_name, mode=self.cfg.mode,
                                       normalize=True)

        self.proj = VariationalProjection(
            in_dim=self.backbone.hidden_size,
            hidden=self.cfg.proj_hidden,
            z_dim=self.cfg.z_dim,
        )
        self.task_head = nn.Sequential(
            nn.Linear(self.cfg.z_dim, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, n_task_classes),
        )
        self.tier_heads = nn.ModuleList([
            TierAdversaryHead(self.cfg.z_dim, t, hidden=self.cfg.adv_hidden)
            for t in tiers
        ])

        if self.cfg.use_mine:
            self.mine = MINE(self.cfg.z_dim,
                             a_dim=attribute_vector_dim(tiers),
                             hidden=self.cfg.mine_hidden)
        else:
            self.mine = None

    # ── Training-time forward ─────────────────────────────────────
    def forward(self, texts: list[str], lambda_adv: float, device: str = "cpu"):
        e = self.backbone(texts, device=device)
        mu, logvar = self.proj(e)
        z = self.proj.reparameterize(mu, logvar)
        z_rev = grad_reverse(z, lambda_adv)
        tier_outs = [head(z_rev) for head in self.tier_heads]
        return {
            "z": z, "mu": mu, "logvar": logvar,
            "task": self.task_head(z),
            "tiers": tier_outs,
        }

    # ── Internal encode (Tensor) ──────────────────────────────────
    def encode(self, texts: list[str], *, device: str = "cpu",
               training: bool = False, dp_sigma: float = 0.0,
               normalize_z: bool = True) -> torch.Tensor:
        e = self.backbone(texts, device=device)
        mu, logvar = self.proj(e)
        z = self.proj.reparameterize(mu, logvar) if training else mu
        if normalize_z:
            z = F.normalize(z, p=2, dim=1)
        if dp_sigma > 0.0:
            z = z + torch.randn_like(z) * dp_sigma
        return z

    # ── Public release API ────────────────────────────────────────
    @torch.no_grad()
    def encode_for_release(self,
                           texts: list[str],
                           *,
                           epsilon: Optional[float] = 2.0,
                           delta: float = 1e-5,
                           device: str = "cpu",
                           batch_size: int = 64) -> np.ndarray:
        """
        Production embedding API.
        - L2-normalizes z so any pair of releases differs by at most 2.0.
        - Adds calibrated Gaussian noise for (epsilon, delta)-DP per release.
        - Pass epsilon=None to skip noise (for utility benchmarking only).
        """
        sigma = (gaussian_release_sigma(epsilon, delta, sensitivity=2.0)
                 if epsilon is not None else 0.0)
        self.eval()
        outs = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i + batch_size]
            z = self.encode(chunk, device=device, training=False,
                            dp_sigma=sigma, normalize_z=True)
            outs.append(z.cpu().numpy())
        return np.concatenate(outs, axis=0)


# ═════════════════════════════════════════════════════════════════
#  LOSS
# ═════════════════════════════════════════════════════════════════

def _kl_unit_normal(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()


def attribute_vector_dim(tiers: list[SensitivityTier]) -> int:
    d = 0
    for tier in tiers:
        for a in tier.attributes:
            d += 1 if a.kind in ("binary", "continuous") else a.n_classes
    return d


def build_attribute_vector(sens: dict[str, torch.Tensor],
                           tiers: list[SensitivityTier]) -> torch.Tensor:
    vecs = []
    for tier in tiers:
        for a in tier.attributes:
            t = sens[a.name]
            if a.kind == "binary":
                vecs.append(t.float().unsqueeze(-1))
            elif a.kind == "categorical":
                vecs.append(F.one_hot(t.long(), num_classes=a.n_classes).float())
            else:
                std = t.float().std().clamp(min=1e-6)
                vecs.append(((t.float() - t.float().mean()) / std).unsqueeze(-1))
    return torch.cat(vecs, dim=-1)


def privacy_loss(out, y_task, sens, tiers, *,
                 lambda_kl, lambda_mine, mine_estimate=None):
    L_task = F.cross_entropy(out["task"], y_task)
    parts = {"task": L_task.item()}

    L_adv_total = torch.tensor(0.0, device=L_task.device)
    for tier, tier_out in zip(tiers, out["tiers"]):
        tier_loss = torch.tensor(0.0, device=L_task.device)

        binary_attrs = [a for a in tier.attributes if a.kind == "binary"]
        if binary_attrs:
            logits = tier_out["binary"]
            targets = torch.stack(
                [sens[a.name].float() for a in binary_attrs], dim=-1,
            )
            pos_w = torch.tensor([a.pos_weight for a in binary_attrs],
                                 device=logits.device)
            tier_loss = tier_loss + F.binary_cross_entropy_with_logits(
                logits, targets, pos_weight=pos_w,
            )
        for a in tier.categorical_attrs:
            tier_loss = tier_loss + F.cross_entropy(
                tier_out["categorical"][a.name], sens[a.name].long(),
            )
        for a in tier.continuous_attrs:
            tier_loss = tier_loss + F.mse_loss(
                tier_out["continuous"][a.name], sens[a.name].float(),
            )

        parts[f"adv:{tier.name}"] = tier_loss.item()
        L_adv_total = L_adv_total + tier.weight * tier_loss

    L_kl = _kl_unit_normal(out["mu"], out["logvar"])
    parts["kl"] = L_kl.item()

    L_mine = (mine_estimate if mine_estimate is not None
              else torch.tensor(0.0, device=L_task.device))
    if mine_estimate is not None:
        parts["mine"] = L_mine.item()

    total = L_task + L_adv_total + lambda_kl * L_kl + lambda_mine * L_mine
    parts["total"] = total.item()
    return total, parts


# ═════════════════════════════════════════════════════════════════
#  DP HELPERS
# ═════════════════════════════════════════════════════════════════

def gaussian_release_sigma(epsilon: float, delta: float,
                           sensitivity: float = 2.0) -> float:
    """
    Sigma for the Gaussian mechanism on a single L2-normalized embedding
    release. With L2-normalization, any two embeddings differ by at most 2.
    """
    return sensitivity * math.sqrt(2.0 * math.log(1.25 / delta)) / epsilon


# ═════════════════════════════════════════════════════════════════
#  DATASET
# ═════════════════════════════════════════════════════════════════

class TieredAttrDataset(Dataset):
    def __init__(self, texts, y_task, sens_dict, tiers):
        self.texts = texts
        self.y_task = torch.tensor(y_task, dtype=torch.long)
        self.sens = {}
        for tier in tiers:
            for a in tier.attributes:
                arr = sens_dict[a.name]
                self.sens[a.name] = (torch.tensor(arr, dtype=torch.float32)
                                     if a.kind == "continuous"
                                     else torch.tensor(arr, dtype=torch.long))

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        return (self.texts[i], self.y_task[i],
                {k: v[i] for k, v in self.sens.items()})


def collate(batch):
    texts = [b[0] for b in batch]
    y_task = torch.stack([b[1] for b in batch])
    keys = batch[0][2].keys()
    sens = {k: torch.stack([b[2][k] for b in batch]) for k in keys}
    return texts, y_task, sens


# ═════════════════════════════════════════════════════════════════
#  TRAINING
# ═════════════════════════════════════════════════════════════════

def train(model: PrivacyEncoder,
          loader: DataLoader,
          *,
          epochs: int = 10,
          device: str = "cpu",
          log_every: int = 1) -> EMA:
    """
    Two-loop adversarial training with KL annealing and EMA tracking.
    Returns the EMA wrapper for deployment-time inference.
    """
    cfg = model.cfg
    model.to(device)

    enc_params = [p for n, p in model.named_parameters()
                  if not n.startswith("mine.") and p.requires_grad]
    mine_params = (list(model.mine.parameters())
                   if model.mine is not None else [])

    opt_main = torch.optim.AdamW(enc_params, lr=cfg.lr_main,
                                 weight_decay=cfg.weight_decay)
    opt_mine = (torch.optim.AdamW(mine_params, lr=cfg.lr_mine)
                if mine_params else None)

    ema = EMA(model, decay=cfg.ema_decay)

    print(f"  encoder params (trainable) : {sum(p.numel() for p in enc_params):,}")
    if mine_params:
        print(f"  MINE params                : {sum(p.numel() for p in mine_params):,}")

    for epoch in range(epochs):
        # adversarial weight: Ganin sigmoid schedule
        p_frac = epoch / max(1, epochs - 1)
        lambda_adv = cfg.lambda_adv_max * (
            2.0 / (1.0 + math.exp(-10 * p_frac)) - 1.0)
        # KL anneal: linear ramp over kl_anneal_epochs
        lambda_kl = cfg.lambda_kl_max * min(
            1.0, (epoch + 1) / max(1, cfg.kl_anneal_epochs))

        agg = {"task": 0.0, "kl": 0.0, "mine": 0.0, "n": 0}
        for tier in model.tiers:
            agg[f"adv:{tier.name}"] = 0.0

        for texts, y_task, y_sens in loader:
            y_task = y_task.to(device)
            y_sens = {k: v.to(device) for k, v in y_sens.items()}

            # ── Inner: train MINE to estimate I(z; A) ──
            if model.mine is not None and opt_mine is not None:
                for _ in range(cfg.mine_steps_per_batch):
                    with torch.no_grad():
                        z_det = model.encode(texts, device=device,
                                             training=True, normalize_z=False)
                    a_vec = build_attribute_vector(y_sens, model.tiers)
                    mi = model.mine(z_det, a_vec)
                    (-mi).backward()
                    opt_mine.step()
                    opt_mine.zero_grad()

            # ── Outer: encoder + tier heads ──
            out = model(texts, lambda_adv, device=device)
            mine_est = None
            if model.mine is not None:
                a_vec = build_attribute_vector(y_sens, model.tiers)
                mine_est = model.mine(out["z"], a_vec)

            loss, parts = privacy_loss(
                out, y_task, y_sens, model.tiers,
                lambda_kl=lambda_kl,
                lambda_mine=cfg.lambda_mine,
                mine_estimate=mine_est,
            )
            opt_main.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc_params, cfg.grad_clip)
            opt_main.step()
            ema.update(model)

            bs = y_task.size(0)
            for k, v in parts.items():
                if k in agg: agg[k] += v * bs
            agg["n"] += bs

        if epoch % log_every == 0 or epoch == epochs - 1:
            n = agg["n"]
            adv_str = " ".join(
                f"{t.name}={agg[f'adv:{t.name}']/n:.3f}"
                for t in model.tiers)
            mine_str = (f"  mine={agg['mine']/n:.3f}"
                        if model.mine is not None else "")
            print(f"  epoch {epoch+1:2d}/{epochs}  "
                  f"λ_adv={lambda_adv:.2f}  λ_kl={lambda_kl:.4f}  "
                  f"task={agg['task']/n:.3f}  kl={agg['kl']/n:.3f}  "
                  f"{adv_str}{mine_str}")

    return ema


# ═════════════════════════════════════════════════════════════════
#  AUDIT (post-hoc adversaries)
# ═════════════════════════════════════════════════════════════════

def audit(model: PrivacyEncoder,
          texts: list[str], y_task: np.ndarray,
          sens: dict[str, np.ndarray],
          *, epsilon: Optional[float] = None, delta: float = 1e-5,
          device: str = "cpu", seed: int = 0) -> dict:
    """
    Post-hoc strong adversary: trains fresh classifiers/regressors per
    sensitive attribute on RELEASED embeddings (with optional DP noise).
    """
    from sklearn.neural_network import MLPClassifier, MLPRegressor
    from sklearn.metrics import accuracy_score, roc_auc_score, r2_score

    Z = model.encode_for_release(
        texts, epsilon=epsilon, delta=delta, device=device,
    )

    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(Z)); cut = len(Z) // 2
    tr, te = idx[:cut], idx[cut:]

    task_clf = MLPClassifier((128, 64), max_iter=400,
                             random_state=seed).fit(Z[tr], y_task[tr])
    task_acc = accuracy_score(y_task[te], task_clf.predict(Z[te]))

    per_attr = {}
    for tier in model.tiers:
        for a in tier.attributes:
            y = sens[a.name]
            if a.kind == "continuous":
                m = MLPRegressor((128, 64), max_iter=400,
                                 random_state=seed).fit(Z[tr], y[tr])
                r2 = r2_score(y[te], m.predict(Z[te]))
                per_attr[a.name] = {"tier": tier.name, "kind": "continuous",
                                    "r2": float(r2),
                                    "severity": _sev_cont(r2)}
            else:
                m = MLPClassifier((128, 64), max_iter=400,
                                  random_state=seed).fit(Z[tr], y[tr])
                pred = m.predict(Z[te])
                acc = accuracy_score(y[te], pred)
                base = max(np.bincount(y[te]) / len(y[te]))
                auc = (roc_auc_score(y[te], m.predict_proba(Z[te])[:, 1])
                       if a.n_classes == 2 else None)
                per_attr[a.name] = {
                    "tier": tier.name, "kind": a.kind,
                    "accuracy": float(acc), "advantage": float(acc - base),
                    "auc": float(auc) if auc is not None else None,
                    "severity": _sev_disc(acc - base, auc),
                }
    return {
        "task_accuracy": float(task_acc),
        "per_attribute": per_attr,
        "tier_summary": _tier_summary(per_attr, model.tiers),
        "epsilon_used": epsilon,
    }


def _sev_disc(adv: float, auc: Optional[float]) -> str:
    s = max(adv, (auc - 0.5) if auc is not None else 0.0)
    if s > 0.30: return "CRITICAL"
    if s > 0.15: return "HIGH"
    if s > 0.05: return "MEDIUM"
    if s > 0.01: return "LOW"
    return "NEGLIGIBLE"


def _sev_cont(r2: float) -> str:
    if r2 > 0.50: return "CRITICAL"
    if r2 > 0.30: return "HIGH"
    if r2 > 0.10: return "MEDIUM"
    if r2 > 0.02: return "LOW"
    return "NEGLIGIBLE"


def _tier_summary(per_attr: dict, tiers) -> dict:
    order = ["NEGLIGIBLE", "LOW", "MEDIUM", "HIGH", "CRITICAL"]
    out = {}
    for tier in tiers:
        sevs = [d["severity"] for d in per_attr.values()
                if d["tier"] == tier.name]
        out[tier.name] = {
            "n_attributes": len(sevs),
            "worst_severity": max(sevs, key=lambda s: order.index(s)) if sevs else "—",
            "n_critical": sum(1 for s in sevs if s == "CRITICAL"),
            "n_high":     sum(1 for s in sevs if s == "HIGH"),
            "n_medium":   sum(1 for s in sevs if s == "MEDIUM"),
            "n_low":      sum(1 for s in sevs if s == "LOW"),
            "n_negligible": sum(1 for s in sevs if s == "NEGLIGIBLE"),
        }
    return out


def print_report(label: str, report: dict):
    print(f"\n  {label}")
    print(f"    epsilon       : {report['epsilon_used']}")
    print(f"    task_accuracy : {report['task_accuracy']:.3f}")
    print(f"    {'tier':<14}{'#attrs':>8}{'worst':>14}"
          f"{'#crit':>7}{'#high':>7}{'#med':>7}{'#low':>7}{'#negl':>7}")
    for name, s in report["tier_summary"].items():
        print(f"    {name:<14}{s['n_attributes']:>8}{s['worst_severity']:>14}"
              f"{s['n_critical']:>7}{s['n_high']:>7}{s['n_medium']:>7}"
              f"{s['n_low']:>7}{s['n_negligible']:>7}")


# ═════════════════════════════════════════════════════════════════
#  DEMO
# ═════════════════════════════════════════════════════════════════

def _make_demo(n: int = 600, seed: int = 0):
    """
    Synthetic clinical-style notes with three sensitivity tiers:
      tier 'legal'    : gender (binary), age_bucket (3-class)
      tier 'contract' : insurance_type (binary), region (4-class)
      tier 'internal' : age_years (continuous), visit_count (continuous)
    Task: outcome sentiment (positive / negative).
    """
    rng = np.random.RandomState(seed)
    pos = ["recovering well", "stable", "improving daily",
           "responding to treatment"]
    neg = ["in pain", "deteriorating", "complications observed",
           "not improving"]
    male = ["he", "him", "his"]
    female = ["she", "her", "hers"]
    age_words = {0: ["young", "in their twenties"],
                 1: ["middle-aged", "in their forties"],
                 2: ["elderly", "in their seventies"]}
    regions = ["Northern", "Eastern", "Southern", "Western"]
    insurance = ["public", "private"]

    sens = {k: [] for k in
            ["gender", "age_bucket", "insurance_type", "region",
             "age_years", "visit_count"]}
    texts, y_task = [], []

    for _ in range(n):
        t = rng.randint(2)
        g = rng.randint(2); ab = rng.randint(3)
        ins = rng.randint(2); reg = rng.randint(4)
        ay = (rng.uniform(20, 30) if ab == 0
              else rng.uniform(40, 55) if ab == 1
              else rng.uniform(65, 85))
        vc = float(rng.poisson(3 + ab))
        adj = rng.choice(pos if t == 1 else neg)
        pron = rng.choice(male if g == 0 else female)
        descriptor = rng.choice(age_words[ab])
        text = (f"Patient note: the {descriptor} patient ({insurance[ins]} "
                f"insurance, {regions[reg]} region) — {pron} is {adj}. "
                f"Visit count: {int(vc)}. Reviewed Mon.")
        texts.append(text); y_task.append(t)
        sens["gender"].append(g); sens["age_bucket"].append(ab)
        sens["insurance_type"].append(ins); sens["region"].append(reg)
        sens["age_years"].append(ay); sens["visit_count"].append(vc)

    sens = {k: np.array(v, dtype=np.float32 if k in
            ("age_years", "visit_count") else np.int64)
            for k, v in sens.items()}
    return texts, np.array(y_task), sens


def main():
    torch.manual_seed(0); np.random.seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    print(f"Base model  : {DEFAULT_MODEL}\n")

    texts, y_task, sens = _make_demo(n=600)

    tiers = [
        SensitivityTier("legal", weight=2.0, attributes=[
            Attribute("gender", n_classes=2, kind="binary", pos_weight=1.0),
            Attribute("age_bucket", n_classes=3, kind="categorical"),
        ]),
        SensitivityTier("contract", weight=1.0, attributes=[
            Attribute("insurance_type", n_classes=2, kind="binary"),
            Attribute("region", n_classes=4, kind="categorical"),
        ]),
        SensitivityTier("internal", weight=0.5, attributes=[
            Attribute("age_years",   kind="continuous"),
            Attribute("visit_count", kind="continuous"),
        ]),
    ]

    cfg = PrivacyConfig(
        z_dim=96, mode="lora",        # change to "head" if peft isn't installed
        use_mine=True, use_ib=True,
        lambda_adv_max=1.0, lambda_kl_max=1e-2, lambda_mine=0.05,
        lr_main=3e-4, lr_mine=5e-4,
    )

    print("=" * 76)
    print(" UPGRADED PRIVACY ENCODER  (MiniLM-L12-v2 + tiered + MINE + IB + DP)")
    print("=" * 76)

    model = PrivacyEncoder(tiers=tiers, n_task_classes=2,
                           model_name=DEFAULT_MODEL, cfg=cfg)
    ds = TieredAttrDataset(texts, y_task, sens, tiers)
    loader = DataLoader(ds, batch_size=32, shuffle=True, collate_fn=collate)

    ema = train(model, loader, epochs=10, device=device)

    # Use EMA weights for evaluation (more stable embeddings at deployment)
    eval_model = copy.deepcopy(model)
    ema.copy_to(eval_model)

    # Audits across DP epsilons
    print("\n" + "=" * 76)
    print(" AUDIT  (post-hoc adversaries on released embeddings)")
    print("=" * 76)

    for eps in [None, 4.0, 2.0, 1.0]:
        label = ("Released without DP noise" if eps is None
                 else f"Released under (eps={eps}, delta=1e-5)-DP")
        rep = audit(eval_model, texts, y_task, sens,
                    epsilon=eps, delta=1e-5, device=device)
        print_report(label, rep)


if __name__ == "__main__":
    main()